<a href="https://colab.research.google.com/github/thefnj/Brief_Processor/blob/main/Brief_Processor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q gradio pdfplumber gspread google-auth
import gradio as gr
import pdfplumber
import uuid
import pandas as pd

# (Assume your Google Sheets auth code is here from earlier)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 89.0 MB/s eta 0:00:00


In [1]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default

# 1. Authenticate
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Try to open the sheet
spreadsheet_url = "https://docs.google.com/spreadsheets/d/1ZGpiqI2QJwtLhAmHuehuz-nf4yN0Zyuih-Kp6Tj-5II/edit#gid=0"

try:
    sh = gc.open_by_url(spreadsheet_url)
    print("✅ CONNECTION SUCCESS!")
    print(f"Connected to: {sh.title}")

    # List the tabs to make sure they match your code
    worksheets = sh.worksheets()
    print(f"Tabs found: {[ws.title for ws in worksheets]}")

except Exception as e:
    print(f"❌ CONNECTION FAILED: {e}")

✅ CONNECTION SUCCESS!
Connected to: Brief_DB
Tabs found: ['Ideas', 'Components', 'Brief_Requests', 'Matches']


In [6]:
!pip install -q pdfplumber
import pdfplumber

# CHANGE THIS to the name of the file you just uploaded
file_path = "your_test_file.pdf"

try:
    with pdfplumber.open(file_path) as pdf:
        # Just grab the first 500 characters of the first page
        first_page = pdf.pages[0]
        text = first_page.extract_text()

        if text:
            print("✅ PDF PLUMBER SUCCESS!")
            print("-" * 30)
            print(f"Snippet: {text[:500]}...")
            print("-" * 30)
        else:
            print("⚠️ PDF opened, but NO TEXT was found. Is the PDF an image/scan?")

except Exception as e:
    print(f"❌ PDF ERROR: {e}")

❌ PDF ERROR: [Errno 2] No such file or directory: 'your_test_file.pdf'


In [5]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Modular Creative Engine")
    gr.Markdown("Upload a specific PDF brief to genericize it and atomize it into components.")

    with gr.Row():
        with gr.Column():
            file_input = gr.File(label="Upload Brief (PDF)")
            process_btn = gr.Button("Analyze & Genericize", variant="primary")

        with gr.Column():
            output_display = gr.Markdown("### AI Analysis Results")
            with gr.Group():
                title_edit = gr.Textbox(label="Edit Generic Title")
                insight_edit = gr.Textbox(label="Edit Core Insight")

            sync_btn = gr.Button("Push to Google Sheets DB")
            status_text = gr.Label(label="Sync Status")

    # Wire up the buttons
    process_btn.click(
        fn=process_brief_flow,
        inputs=file_input,
        outputs=[output_display, title_edit, insight_edit]
    )

    sync_btn.click(
        fn=sync_to_db,
        inputs=[title_edit, insight_edit], # Add your ID field here too
        outputs=status_text
    )

demo.launch(debug=True)

/tmp/ipykernel_2729/2767560171.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


NameError: name 'process_brief_flow' is not defined

In [7]:
def process_brief_flow(pdf_file):
    # 1. Extract Text from the uploaded PDF
    with pdfplumber.open(pdf_file.name) as pdf:
        raw_text = " ".join([page.extract_text() for page in pdf.pages if page.extract_text()])

    # 2. The "AI" Step (Tonight: Simple extraction / Tomorrow: Gemini API)
    # For tonight, we'll simulate the AI output so you can test the UI
    generic_data = {
        "id": f"IDEA-{uuid.uuid4().hex[:4].upper()}",
        "title": "Generic High-Engagement Campaign",
        "sector": "Retail / FMCG",
        "insight": "Consumers seek convenience over brand loyalty during holiday peaks.",
        "components": ["Social Video", "Digital Display", "In-store POS"]
    }

    # 3. Format strings for the UI display
    display_text = f"**ID:** {generic_data['id']}\n**Concept:** {generic_data['title']}\n**Insight:** {generic_data['insight']}"

    return display_text, generic_data['title'], generic_data['insight']

def sync_to_db(id_val, title, insight):
    # This is where your gspread code from earlier triggers
    # ideas_sheet.append_row([id_val, title, insight, "Source: PDF"])
    return f"Successfully pushed {id_val} to Google Sheets!"

In [8]:
def process_brief_to_db(pdf_path, public_source_csv):
    # 1. Load your Public Source for the "Tie Back"
    df_public = pd.read_csv(public_source_csv)

    # 2. Extract Text from PDF
    with pdfplumber.open(pdf_path) as pdf:
        full_text = ""
        for page in pdf.pages:
            full_text += page.extract_text()

    # 3. MAPPING LOGIC (The "Tie Back")
    # Search the PDF text for a Campaign Title from your Public Source
    matched_row = None
    for index, row in df_public.iterrows():
        if row['Internal_Campaign_Name'] in full_text:
            matched_row = row
            break

    # 4. FORMULATE THE NEW 'IDEA' ENTRY
    new_idea = {
        "Idea_ID": f"IDEA-{uuid.uuid4().hex[:4].upper()}",
        "Generic_idea_title": matched_row['Generic_Campaign_Concept'] if matched_row is not None else "Unknown Concept",
        "Source_deck": pdf_path.split('/')[-1],
        "Original_sector": matched_row['Generic_Brand_Category'] if matched_row is not None else "General Retail",
        "Summary": matched_row['Generic_Key_Objective'] if matched_row is not None else "Extracted from PDF",
        "Public_Source_Ref": matched_row['ID'] if matched_row is not None else "NEW-ENTRY"
    }

    return new_idea


In [9]:
def process_brief_to_db(pdf_path, public_source_csv):
    # 1. Load your Public Source for the "Tie Back"
    df_public = pd.read_csv(public_source_csv)

    # 2. Extract Text from PDF
    with pdfplumber.open(pdf_path) as pdf:
        full_text = ""
        for page in pdf.pages:
            full_text += page.extract_text()

    # 3. MAPPING LOGIC (The "Tie Back")
    # Search the PDF text for a Campaign Title from your Public Source
    matched_row = None
    for index, row in df_public.iterrows():
        if row['Internal_Campaign_Name'] in full_text:
            matched_row = row
            break

    # 4. FORMULATE THE NEW 'IDEA' ENTRY
    new_idea = {
        "Idea_ID": f"IDEA-{uuid.uuid4().hex[:4].upper()}",
        "Generic_idea_title": matched_row['Generic_Campaign_Concept'] if matched_row is not None else "Unknown Concept",
        "Source_deck": pdf_path.split('/')[-1],
        "Original_sector": matched_row['Generic_Brand_Category'] if matched_row is not None else "General Retail",
        "Summary": matched_row['Generic_Key_Objective'] if matched_row is not None else "Extracted from PDF",
        "Public_Source_Ref": matched_row['ID'] if matched_row is not None else "NEW-ENTRY"
    }

    return new_idea

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

# Authorize the gspread client
gc = gspread.authorize(creds)

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

# Authorize gspread
gc = gspread.authorize(creds)

# Your specific Spreadsheet URL
spreadsheet_url = "https://docs.google.com/spreadsheets/d/1ZGpiqI2QJwtLhAmHuehuz-nf4yN0Zyuih-Kp6Tj-5II/edit#gid=0"

try:
    spreadsheet = gc.open_by_url(spreadsheet_url)

    # Accessing all four tabs from your screenshot
    requests_sheet = spreadsheet.worksheet("Brief_Requests")
    ideas_sheet    = spreadsheet.worksheet("Ideas")
    components_sheet = spreadsheet.worksheet("Components")
    matches_sheet  = spreadsheet.worksheet("Matches")

    print("✅ Successfully connected to all four database tabs.")

except Exception as e:
    print(f"❌ Error: Could not connect to the sheet. Check permissions or tab names. \nDetails: {e}")

# --- QUICK HELPER FUNCTION FOR THE DEMO ---
def add_row_to_tab(sheet_object, data_list):
    """Appends a list of values as a new row to the specified tab."""
    sheet_object.append_row(data_list)
    print(f"Successfully added entry to {sheet_object.title}")

✅ Successfully connected to all four database tabs.


In [ ]:
def push_to_sheets(data_dict, target_worksheet):
    """
    Takes a dictionary and appends it as a new row to the target Google Sheet.
    Assumes the dictionary keys match the order of your columns.
    """
    # Convert dictionary values to a list in the correct order
    row_to_append = list(data_dict.values())

    # Append the row
    target_worksheet.append_row(row_to_append)
    print(f"Sync Complete: Data pushed to {target_worksheet.title}")

# Example usage during the demo:
# push_to_sheets(new_generic_idea, ideas_sheet)

In [ ]:
def push_components_to_sheets(components_list, components_sheet):
    for comp in components_list:
        row = list(comp.values())
        components_sheet.append_row(row)
    print(f"Sync Complete: {len(components_list)} components added.")

In [10]:
# 1. Install necessary libraries
!pip install -q -U transformers accelerate bitsandbytes

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# 2. Load a lightweight, smart model (Gemma 2B is great for this)
model_id = "google/gemma-1.1-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
# We use 4-bit quantization to make sure it fits in Colab's memory
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("✅ Local Model Loaded & Ready")

# 3. The Local Processing Function
def process_brief_locally(brief_text):
    prompt = f"Extract the core concept and remove brand names from this brief. Format as JSON: {brief_text}"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=200)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.2 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-1.1-2b-it.
401 Client Error. (Request ID: Root=1-6a06e1af-4ab759f933ca732e4f7b183d;7db61491-0562-4518-95b4-6d04a52430a6)

Cannot access gated repo for url https://huggingface.co/google/gemma-1.1-2b-it/resolve/main/config.json.
Access to model google/gemma-1.1-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
#@title 🚀 Brief Processor Input
#@markdown Enter the brief details below to process them into the DB.

campaign_name = "Nike Run Club" #@param {type:"string"}
sector = "Retail" #@param ["Retail", "Tech", "Finance", "Auto"]
budget = 50000 #@param {type:"slider", min:10000, max:1000000, step:10000}

# Your logic runs when you play the cell
print(f"Processing {campaign_name} for the {sector} sector...")